# AML intensive treatment infection prediction model -- Data preprocessing

data used (without imputation)
- data_final1_wo_imputation.csv 

In [71]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

pd.set_option('display.max_columns', None)

Data preprocessing for all of these

In [72]:
interpolation = pd.read_csv('/path/to/data_final1_wo_interpolation.csv')
uncertainity = pd.read_csv('/path/to/data_final1_uncertainty.csv')
NOimputation = pd.read_csv('/path/to/data_final1_wo_imputation.csv')

/tmp/ipykernel_14896/50879088.py:1: DtypeWarning: Columns (47,99,100) have mixed types. Specify dtype option on import or set low_memory=False.
  interpolation = pd.read_csv('/path/to/data_final1_wo_interpolation.csv')
/tmp/ipykernel_14896/50879088.py:2: DtypeWarning: Columns (48,100,101) have mixed types. Specify dtype option on import or set low_memory=False.
  uncertainity = pd.read_csv('/path/to/data_final1_uncertainty.csv')
/tmp/ipykernel_14896/50879088.py:3: DtypeWarning: Columns (47,99,100) have mixed types. Specify dtype option on import or set low_memory=False.
  NOimputation = pd.read_csv('/path/to/data_final1_wo_imputation.csv')


## DATA

In [73]:
# remove these columns from the data
delete = pd.read_excel('/path/to/data_final_dg_documentation0.xlsx')
delist = delete[delete['poista_analyyseista']==1]['names'][2:].tolist()
delist.remove('treatment_date')
delist.remove('naytteenotto_hetki')
delist

['infection',
 'time_to_infection',
 'source',
 'time_diff_mgg_treatmentstart',
 'syntymaaika_pvm',
 'effect_end_date',
 'infection_date',
 'first_treatment_date',
 'dg_date_combined',
 'etiology',
 'time_diff_lab_treatment_date',
 'neutropenia_days_005',
 'neutropenia_days_050']

In [74]:
# differently imputed data to try
dflist = [interpolation, uncertainity, NOimputation]
dfnames = ['interpolation', 'uncertainty', 'NOimputation']

In [75]:
len(interpolation) # data_final1_wo_interpolation.csv
len(uncertainity) # data_final1_uncertainty.scv
len(NOimputation) # data_final1_wo_imputation.csv

5946

14314

7301

remove treatment cycles from NOimputation that have over 85% NAns in specific columns

In [76]:
cols = ["temperature", "b_neut"]   # whatever columns you care about
threshold = 0.85

bad_groups = set()   # will store (henkilotunnus, cycle_number) pairs

for col in cols:
    frac_nan_per_group = (
        NOimputation.groupby(["henkilotunnus", "cycle_number"])[col]
                    .apply(lambda s: s.isna().mean())
    )

    # groups where this column has ≥ 85% NaNs
    bad_here = frac_nan_per_group[frac_nan_per_group >= threshold]

    print(f"\nColumn: {col}")
    print("Number of groups to drop:", len(bad_here))
    #print(bad_here)          # optional: see which ones

    # add these groups to the global set
    bad_groups.update(bad_here.index)

BAD_GROUPS = list(bad_groups)


Column: temperature
Number of groups to drop: 0

Column: b_neut
Number of groups to drop: 0


## Function

In [77]:

def process_data(df, dfname, delist):
    '''  Processes the dataframe for the infection prediction model  '''
    
    print(f"-------------- {dfname} --------------")
    inf_cycles_num = df.groupby(['henkilotunnus', 'cycle_number']).count()
    print("Cycles before preprocessing:", len(inf_cycles_num))

    # create countdown col from the starting day of the treatment
    #df['countdown'] = df.groupby(['henkilotunnus', 'cycle_number']).cumcount()
    # make sure it's datetime
    df["naytteenotto_hetki"] = pd.to_datetime(df["naytteenotto_hetki"])

    g = df.groupby(["henkilotunnus", "cycle_number"])["naytteenotto_hetki"]

    # 0-based day offset from the earliest timestamp in the group
    df["countdown"] = (df["naytteenotto_hetki"] - g.transform("min")).dt.days
    #df[['henkilotunnus', 'naytteenotto_hetki', 'infektion_binary', 'cycle_number', 'countdown_days']].head(20)

    # delete unneccessary columns
    print('Cols before del:', len(df.columns))
    df = df.drop(columns=delist, errors='ignore')
    print('Cols after del:', len(df.columns))
    print()
    
    # categorical data to int
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    categorical_cols.remove('henkilotunnus')
    categorical_cols.remove('treatment_date')

    #categorical_cols.remove('naytteenotto_hetki')
    categorical_cols = list(set(categorical_cols) - set(delist))
    categorical = ['ELN', 'sukupuoli_selite', 'sykli']
    
    # categorical data to dymmy
    df = pd.get_dummies(df, columns = categorical, dtype=float) 
    df = df.drop(columns=['sykli_KONS', 'sukupuoli_selite_Nainen'])

    # replace true-false with 0 and 1
    non_cat = list(set(categorical_cols) - set(categorical))
    if 'naytteenotto_hetki' in non_cat:
        non_cat.remove('naytteenotto_hetki')
    df[non_cat] = df[non_cat].apply(pd.to_numeric, errors='coerce')
    
    # Check for infinity values
    is_infinite = df.isin([np.inf, -np.inf])
    res = is_infinite.any().any()
    print('Does df contain infinite values:', res)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    print(df.groupby(["henkilotunnus", "cycle_number"]).ngroups)
    
    # mark groups that should be dropped
    drop_group = (
        df.groupby(["henkilotunnus", "cycle_number"])
            .apply(lambda g: ((g["countdown"] < 6) & (g["fn_day"] == 1)).any())
    )

    # keep only groups that are NOT marked for dropping
    df = df[~df.set_index(["henkilotunnus", "cycle_number"]).index.isin(drop_group[drop_group].index)]
    print(df.groupby(["henkilotunnus", "cycle_number"]).ngroups)
    
    # -------------------------- Drop bad groups -------------------------
    print("--> groups that have over 85% of nan values (in b_neut or temperature) dropped:", len(BAD_GROUPS))
    # Build a boolean mask: True if this row belongs to a bad group
    mask_bad = df[["henkilotunnus", "cycle_number"]] \
        .apply(tuple, axis=1) \
        .isin(BAD_GROUPS)

    # Keep only rows that are NOT in bad groups
    df = df[~mask_bad]
    
    # --------------------------------------------------------------------

    df['transfusion_dependent_6x'] = df['transfusion_dependent_6x'].astype(float)
    

    # lastly lets only keep adult patients

    # BEFORE filtering, save the original unique IDs
    orig_ids = set(df['henkilotunnus'].unique())
    
    # Coerce to numeric just in case
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    
    # Keep only IDs whose minimum age across all their rows is >= 18
    valid_mask = df.groupby('henkilotunnus')['age'].transform('min') >= 18
    df = df[valid_mask].copy()

    # ... after your filtering step to keep adults only ...
    kept_ids = set(df['henkilotunnus'].unique())
    
    discarded_count = len(orig_ids - kept_ids)
    print('--> Discarded pediatric patients:', discarded_count)

    # statistics
    print()
    inf_cycles = df.groupby(['henkilotunnus', 'cycle_number'])['infektion_binary'].sum().gt(0).mean() * 100
    inf_cycles_num = df.groupby(['henkilotunnus', 'cycle_number']).count()
    print("Number of patients:", len(df['henkilotunnus'].drop_duplicates()))
    print("Number of cycles:", len(inf_cycles_num))
    print(f"Infection cycles (%): {inf_cycles}\n")
    pr = (len(df[df['infektion_binary']==1]) / len(df[df['infektion_binary']==0]) )
    print("Amount of infections:", pr)
    print("Number of columns:", len(df.columns))
    print('number of rows', len(df))
    
    #print("\n-----------------------------------------\n")
    print('\n')

    return df

In [78]:
df_dict = {}
for i, (df, name) in enumerate(zip(dflist, dfnames)):
    df_dict[name] = process_data(df, name, delist)

-------------- interpolation --------------
Cycles before preprocessing: 639
Cols before del: 288
Cols after del: 275

Does df contain infinite values: True
639
624
--> groups that have over 85% of nan values (in b_neut or temperature) dropped: 0
--> Discarded pediatric patients: 0

Number of patients: 269
Number of cycles: 624
Infection cycles (%): 66.34615384615384

Amount of infections: 0.18196489812386524
Number of columns: 277
number of rows 5859


-------------- uncertainty --------------
Cycles before preprocessing: 900
Cols before del: 418
Cols after del: 405

Does df contain infinite values: True
900
754
--> groups that have over 85% of nan values (in b_neut or temperature) dropped: 0
--> Discarded pediatric patients: 30

Number of patients: 275
Number of cycles: 662
Infection cycles (%): 68.58006042296071

Amount of infections: 0.1259959236612933
Number of columns: 407
number of rows 12154


-------------- NOimputation --------------
Cycles before preprocessing: 889
Cols befo

In [80]:
remove_list = pd.read_csv("/path/to/exclude_vars.csv")['variables'].tolist()

In [82]:
for name in df_dict: 
    rmv_list = list(set(remove_list) & set(df_dict[name].columns.tolist()))
    df_dict[name].drop(columns=rmv_list, inplace=True)
    df_dict[name].to_csv(f"/path/to/model_data_{name}.csv")
    len(df_dict[name].columns.tolist())


277

382

277

In [83]:
df_dict['NOimputation'][['henkilotunnus', 'cycle_number']].drop_duplicates().to_csv("/path/to/model_treatment_cycles.csv")

## TESTEJÄ

In [86]:
# check the sizes of the cycle lengths
sizes = df_dict['NOimputation'].groupby(['henkilotunnus', 'cycle_number']).size().reset_index()

len(sizes[sizes[0]==1])
len(sizes[sizes[0]==2])
len(sizes[sizes[0]==3])
len(sizes[sizes[0]==4])
len(sizes[sizes[0]==5])
len(sizes[sizes[0]==6])
len(sizes[sizes[0]==7])
len(sizes[sizes[0]==8])

3

9

17

31

59

68

85

72

In [ ]:
df_dict['uncertainty'].describe()

,cycle_number,infektion_binary,ERY_vacuolated_mild,ERY_large_ery_mild,ERY_megaloblast_mild,ERY_dysmorphic_mild,ERY_multinucleated_mild,APL_auer,APL_nucleus,MEG_separated,MEG_large,MEG_hyperlob,MEG_hypolob,MGK_count,cellularity,ERY_dysplasia,APL_dysplasia,MGK_dysplasia,bm_blast_p,age,n_of_previous_infections,n_of_previous_cycles,naytteenotto_hetki,bneut_accumulated_max_005_cycle,bneut_accumulated_max_050_cycle,bneut_accumulated_max_005_total,bneut_accumulated_max_050_total,temperature,uncertainty_havainto_arvo,temperature_mean_last_values_1_days,temperature_trend_from_1_d,temperature_mean_last_values_2_days,temperature_trend_from_2_d,transfusion_dependent_6x,comorbidity_age,komorbiditeetti,DM,comorbidity_wo_age,Cardiovascular_disease,ASXL1,BCOR,CBFB-MYH11,CBL,CEBPA,DNMT3A,EP300,FLT3_ITD,FLT3_TKD,GATA2,IDH1,IDH2,KIT,KRAS,NF1,NPM1,NRAS,PTPN11,RUNX1,RUNX1-RUNX1T1,SRSF2,STAG2,TET2,TP53,WT1,No_mutations,FLT3_mutations,IDH_mutations,CpG_mutations,Transcription_mutations,Spliceosome_mutations,MDS_related_mutations,chromatin_cohesin_mutations,RTK_RAS_mutations,GATA_mutations,RAS_mutations,KMT2A_rearrangement,monosomy7,trisomy21,trisomy8,del_5q,del_7q,inv_16,No_chrms,t_11q_3,t_8_21,complex_karyotype,monosomal_karyotype,translocation3,translocation11,monosomy7_del_7q,monosomy17_del_17p,monosomy5_del_5q,tp53_chr17,tp53_chr17_complex,asxl1_chr20,t1517,t821,inv16,rare,npm1,myelodysplasia,myelodysplasia_related,therapy,tmp,b_baso,b_eos,b_erblast,b_eryt,b_hb,b_hkr,b_leuk,b_ly,b_monos,b_neut,b_trom,e_mch,e_mchc,e_mcv,e_rdw,l_baso,l_eos,l_lymf,l_mono,l_neut,p_afos,p_alat,p_alb,p_crp,p_fi_dd,p_gluk,p_k,p_krea,p_ld,p_mg,p_na,p_pi,p_tt,s_ca_ion,s_p_h,uncertainty_b_baso,uncertainty_b_eos,uncertainty_b_erblast,uncertainty_b_eryt,uncertainty_b_hb,uncertainty_b_hkr,uncertainty_b_leuk,uncertainty_b_ly,uncertainty_b_monos,uncertainty_b_neut,uncertainty_b_trom,uncertainty_e_mch,uncertainty_e_mchc,uncertainty_e_mcv,uncertainty_e_rdw,uncertainty_l_baso,uncertainty_l_blast,uncertainty_l_eos,uncertainty_l_lymf,uncertainty_l_metam,uncertainty_l_mono,uncertainty_l_myelos,uncertainty_l_neut,uncertainty_na,uncertainty_p_afos,uncertainty_p_alat,uncertainty_p_alb,uncertainty_p_crp,uncertainty_p_gluk,uncertainty_p_k,uncertainty_p_krea,uncertainty_p_ld,uncertainty_p_mg,uncertainty_p_na,uncertainty_p_pi,uncertainty_p_tt,uncertainty_s_ca_ion,uncertainty_s_p_h,b_baso_mean_last_values_3_days,b_eos_mean_last_values_3_days,b_erblast_mean_last_values_3_days,b_eryt_mean_last_values_3_days,b_hb_mean_last_values_3_days,b_hkr_mean_last_values_3_days,b_leuk_mean_last_values_3_days,b_ly_mean_last_values_3_days,b_monos_mean_last_values_3_days,b_neut_mean_last_values_3_days,b_trom_mean_last_values_3_days,e_mch_mean_last_values_3_days,e_mchc_mean_last_values_3_days,e_mcv_mean_last_values_3_days,e_rdw_mean_last_values_3_days,l_baso_mean_last_values_3_days,l_blast_mean_last_values_3_days,l_eos_mean_last_values_3_days,l_lymf_mean_last_values_3_days,l_metam_mean_last_values_3_days,l_mono_mean_last_values_3_days,l_myelos_mean_last_values_3_days,l_neut_mean_last_values_3_days,p_afos_mean_last_values_3_days,p_alat_mean_last_values_3_days,p_alb_mean_last_values_3_days,p_crp_mean_last_values_3_days,p_fi_dd_mean_last_values_3_days,p_gluk_mean_last_values_3_days,p_k_mean_last_values_3_days,p_krea_mean_last_values_3_days,p_ld_mean_last_values_3_days,p_mg_mean_last_values_3_days,p_na_mean_last_values_3_days,p_pi_mean_last_values_3_days,p_tt_mean_last_values_3_days,s_ca_ion_mean_last_values_3_days,s_p_h_mean_last_values_3_days,b_baso_trend_from_3_d,b_eos_trend_from_3_d,b_erblast_trend_from_3_d,b_eryt_trend_from_3_d,b_hb_trend_from_3_d,b_hkr_trend_from_3_d,b_leuk_trend_from_3_d,b_ly_trend_from_3_d,b_monos_trend_from_3_d,b_neut_trend_from_3_d,b_trom_trend_from_3_d,e_mch_trend_from_3_d,e_mchc_trend_from_3_d,e_mcv_trend_from_3_d,e_rdw_trend_from_3_d,l_baso_trend_from_3_d,l_eos_trend_from_3_d,l_lymf_trend_from_3_d,l_mono_trend_from_3_d,l_neut_trend_from_3_d,p_afos_trend_from_3_d,p_alat_trend_from_3_d